# Fase 4: modelos globales de aprendizaje automático

Este notebook deja constancia de cómo se evalúan dos **modelos globales** (Ridge y gradient boosting)
frente a las líneas base de la fase 3, con el mismo marco de origen móvil y las mismas series elegibles.

Todas las celdas llaman a funciones del paquete `prevision` y no tienen lógica propia, así que el
notebook y el pipeline (`python -m prevision.pipeline`) no pueden divergir. La última celda lo comprueba.

**Pregunta:** ¿un modelo global supera a la deriva y al ingenuo en la evaluación de origen móvil, y dónde?
Un "no" es un resultado válido.

**Qué es un modelo global.** No recibe una sola serie, sino todas. En cada origen `t` se ajusta con todos
los ejemplos (serie, origen `o`) cuyo curso objetivo `o+1` ya es conocido en `t` (`o+1 <= t`) y prevé `t+1`
para cada serie. Predice el **crecimiento logarítmico** `ln(y[t+1] / y[t])`; la previsión es
`y[t] * exp(predicción)`, de modo que un crecimiento 0 es el ingenuo. Las variables se calculan solo con
cursos con dato hasta el origen, incluidas las medias entre series.

**Reglas fijadas antes de ver resultados:**

- Hiperparámetros fijos, escritos en `config.py` y **nunca ajustados con los pares de evaluación**.
- Versión principal: `ridge` y `gbm` con las variables `base`. Las variantes `*_region` y `*_no_system`
  son **análisis de sensibilidad** y se rotulan así.
- Criterio de éxito: un modelo gana si su WAPE total es menor que el de la deriva **y** no es peor que el
  ingenuo en los tramos mediano y pequeño. Si no, no supera el listón.

In [1]:
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display

from prevision.config import (
    GBM_PARAMS,
    LINKS_FILE,
    MIN_FEATURE_HISTORY,
    MIN_TRAIN,
    MIN_TRAIN_EXAMPLES,
    RAW_FILE,
    RECENT_TARGET_FROM,
    RIDGE_ALPHA,
)
from prevision.evaluation import (
    GLOBAL_MODELS,
    add_region,
    add_size_tier,
    compare_to_naive,
    compute_metrics,
    metrics_report,
    recent_metrics,
    rolling_origin_forecasts,
    rolling_origin_global,
)
from prevision.features import FEATURE_SETS
from prevision.pipeline import build_series, evaluate_models, run_pipeline

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

# Los ficheros del notebook van a una carpeta temporal: no tocan data/processed/.
out_dir = Path(tempfile.mkdtemp())
panel = run_pipeline(RAW_FILE, out_dir)
_, series, quality = build_series(panel, LINKS_FILE, out_dir)
forecasts, metrics = evaluate_models(series, quality, out_dir)

MAIN_MODELS = ["naive", "drift", "ridge", "gbm"]
main_forecasts = forecasts[forecasts["model"].isin(MAIN_MODELS)]

## 1. Variables, hiperparámetros y pares evaluados

Las variables (todas conocidas en el origen) están en `FEATURE_SETS`. No hay identificador de universidad
ni de centro: con tantas series y tan pocos ejemplos dejaría al modelo memorizar series.

In [2]:
for name, columns in FEATURE_SETS.items():
    print(f"{name}: {columns}")
print()
print(f"RIDGE_ALPHA={RIDGE_ALPHA}")
print(f"GBM_PARAMS={GBM_PARAMS}")
print(f"MIN_FEATURE_HISTORY={MIN_FEATURE_HISTORY}, MIN_TRAIN={MIN_TRAIN}, MIN_TRAIN_EXAMPLES={MIN_TRAIN_EXAMPLES}")
print()
print("Modelos globales:", {name: feature_set for name, (_, feature_set) in GLOBAL_MODELS.items()})

base: ['g1', 'g2', 'drift_rel', 'log_size', 'n_seasons', 'pct_women', 'system_growth']
with_region: ['g1', 'g2', 'drift_rel', 'log_size', 'n_seasons', 'pct_women', 'system_growth', 'region_growth']
without_system: ['g1', 'g2', 'drift_rel', 'log_size', 'n_seasons', 'pct_women']

RIDGE_ALPHA=10.0
GBM_PARAMS={'max_iter': 100, 'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 20, 'l2_regularization': 1.0, 'random_state': 0}
MIN_FEATURE_HISTORY=3, MIN_TRAIN=4, MIN_TRAIN_EXAMPLES=30

Modelos globales: {'ridge': 'base', 'gbm': 'base', 'ridge_region': 'with_region', 'gbm_region': 'with_region', 'ridge_no_system': 'without_system', 'gbm_no_system': 'without_system'}


Los modelos globales se evalúan sobre **exactamente los mismos pares** que las líneas base: si un modelo no
puede prever un par se informa con un error, no se rellena con otro modelo.

In [3]:
pairs_per_model = forecasts.groupby("model").size().rename("pares").to_frame().T
display(pairs_per_model)
print(f"{forecasts['series_id'].nunique()} series con pares, de {int(quality['eligible'].sum())} elegibles")

model,drift,gbm,gbm_no_system,gbm_region,moving_average,naive,ridge,ridge_no_system,ridge_region
pares,761,761,761,761,761,761,761,761,761


110 series con pares, de 112 elegibles


## 2. Métricas totales

Mismas métricas que en la fase 3 (MAE, WAPE, MAPE y sesgo; `*_vs_naive` negativo = mejor que el ingenuo).
Se informa con todos los pares y sin los pares cuyo objetivo es 2025-26 (provisional).

**Los 761 pares salen de 110 series y no son independientes** (una serie aporta hasta 7 pares correlacionados).
Las diferencias de décimas de punto entre modelos **no están contrastadas**: no hay intervalo de confianza ni test.

In [4]:
cols = ["model", "n_pairs", "mae", "wape", "mape", "bias", "mae_vs_naive", "wape_vs_naive", "mape_vs_naive"]
for scope in ("todos", "sin_provisional"):
    print(f"Pares: {scope}")
    selected = metrics[(metrics["scope"] == scope) & metrics["target_year"].isna()]
    display(selected[selected["model"].isin(MAIN_MODELS)][cols].round(4))

Pares: todos


,model,n_pairs,mae,wape,mape,bias,mae_vs_naive,wape_vs_naive,mape_vs_naive
0,drift,761,20.5013,0.0430,0.0471,-4.7972,-0.0873,-0.0873,-0.0630
1,gbm,761,19.2323,0.0404,0.0461,0.2942,-0.1438,-0.1438,-0.0827
5,naive,761,22.4625,0.0471,0.0503,-12.3548,0.0000,0.0000,0.0000
6,ridge,761,20.1199,0.0422,0.0458,-5.7840,-0.1043,-0.1043,-0.0891


Pares: sin_provisional


,model,n_pairs,mae,wape,mape,bias,mae_vs_naive,wape_vs_naive,mape_vs_naive
72,drift,651,19.9799,0.0425,0.0468,-4.9233,-0.0909,-0.0909,-0.0545
73,gbm,651,19.1181,0.0407,0.0460,1.3160,-0.1301,-0.1301,-0.0701
77,naive,651,21.9785,0.0467,0.0495,-12.0799,0.0000,0.0000,0.0000
78,ridge,651,20.3937,0.0434,0.0464,-6.3593,-0.0721,-0.0721,-0.0612


### WAPE por curso objetivo

Los primeros cursos objetivo son los menos fiables: en el origen 2018 solo hay un centenar de ejemplos de
entrenamiento. Si un modelo solo gana en cursos concretos, la mejora no es robusta.

In [5]:
by_year = metrics[(metrics["scope"] == "todos") & metrics["target_year"].notna()]
by_year = by_year[by_year["model"].isin(MAIN_MODELS)]
display(by_year.pivot(index="target_year", columns="model", values="wape").round(4))

model,drift,gbm,naive,ridge
target_year,,,,
2019.0,0.0370,0.0393,0.0406,0.0648
2020.0,0.0429,0.0396,0.0428,0.0434
2021.0,0.0384,0.0340,0.0431,0.0367
2022.0,0.0502,0.0514,0.0540,0.0469
2023.0,0.0467,0.0402,0.0493,0.0373
2024.0,0.0394,0.0392,0.0497,0.0334
2025.0,0.0459,0.0387,0.0492,0.0360


## 3. ¿La mejora la sostienen muchos centros o unos pocos?

`compare_to_naive` cuenta en cuántos pares y en cuántas series un modelo tiene menos error absoluto que el
ingenuo.

In [6]:
display(pd.DataFrame({m: compare_to_naive(main_forecasts, m) for m in ("drift", "ridge", "gbm")}).round(3))

,drift,ridge,gbm
n_pairs,761.000,761.000,761.000
share_pairs_better,0.489,0.463,0.485
share_pairs_equal,0.009,0.000,0.000
n_series,110.000,110.000,110.000
n_series_better,44.000,39.000,48.000


## 4. Por tamaño de centro

Terciles de la matrícula en el último curso (`size_last`). Solo descriptivo: el tramo usa un dato de 2025-26
para agrupar, no para prever. Aquí se aplica la segunda parte del criterio de éxito: no ser peor que el
ingenuo en los tramos mediano y pequeño.

In [7]:
tiers = add_size_tier(main_forecasts, quality)
by_tier = compute_metrics(tiers, by=["size_tier"])
display(by_tier[["size_tier", "model", "n_pairs", "mae", "wape", "mape", "bias", "wape_vs_naive"]].round(4))

,size_tier,model,n_pairs,mae,wape,mape,bias,wape_vs_naive
0,grande,drift,294,31.9066,0.0446,0.0481,-12.7280,-0.1756
1,grande,gbm,294,29.4936,0.0413,0.0444,-5.0305,-0.2379
2,grande,naive,294,38.7007,0.0541,0.0613,-28.2585,0.0000
3,grande,ridge,294,31.1304,0.0435,0.0468,-18.1440,-0.1956
4,mediano,drift,291,15.2470,0.0395,0.0407,-1.6364,0.0845
5,mediano,gbm,291,13.2085,0.0343,0.0347,2.0242,-0.0605
6,mediano,naive,291,14.0584,0.0365,0.0379,-4.7732,0.0000
7,mediano,ridge,291,14.5551,0.0377,0.0382,-0.0189,0.0353
8,pequeño,drift,176,10.1366,0.0443,0.0562,3.2248,0.0979
9,pequeño,gbm,176,12.0509,0.0527,0.0678,6.3286,0.3052


## 5. Solo cursos objetivo desde 2021

Subconjunto con varias cohortes de entrenamiento (`RECENT_TARGET_FROM`). **Descriptivo**: se elige el corte
por el tamaño del entrenamiento y no por los resultados; reduce los pares y por eso no sustituye al total.

In [8]:
print(f"Curso objetivo >= {RECENT_TARGET_FROM}")
display(recent_metrics(main_forecasts)[["model", "n_pairs", "mae", "wape", "mape", "bias", "wape_vs_naive"]].round(4))

Curso objetivo >= 2021


,model,n_pairs,mae,wape,mape,bias,wape_vs_naive
0,drift,547,21.4810,0.0441,0.0478,-5.1011,-0.1016
1,gbm,547,19.8087,0.0407,0.0460,-0.6030,-0.1715
2,naive,547,23.9104,0.0491,0.0514,-13.5302,0.0000
3,ridge,547,18.4854,0.0380,0.0427,-2.4639,-0.2269


## 6. Análisis de sensibilidad (no es la versión principal)

> **Rótulo: sensibilidad.** Las variantes `*_region` (añade el crecimiento medio del resto de la comunidad) y
> `*_no_system` (quita el crecimiento medio de todas las series) **no se usan para el veredicto**.
>
> El aporte de `system_growth` se vio **después** de ver los resultados de la versión base. Elegir la variante
> por sus pares de evaluación sería ajustar con los datos de evaluación, así que la versión principal sigue
> siendo `base` aunque otra salga mejor.

El diagnóstico previo daba correlación −0,09 entre el crecimiento medio nacional en `t` y el crecimiento en `t+1`
(+0,03 para la comunidad). Esta tabla mide lo que aportan en los modelos.

In [9]:
sensitivity = metrics[(metrics["scope"] == "todos") & metrics["target_year"].isna()]
display(sensitivity[["model", "n_pairs", "mae", "wape", "mape", "bias", "wape_vs_naive"]].round(4))

,model,n_pairs,mae,wape,mape,bias,wape_vs_naive
0,drift,761,20.5013,0.0430,0.0471,-4.7972,-0.0873
1,gbm,761,19.2323,0.0404,0.0461,0.2942,-0.1438
2,gbm_no_system,761,18.6727,0.0392,0.0452,0.0341,-0.1687
3,gbm_region,761,19.3323,0.0406,0.0467,0.6449,-0.1394
4,moving_average,761,36.9089,0.0775,0.0834,-23.6281,0.6431
5,naive,761,22.4625,0.0471,0.0503,-12.3548,0.0000
6,ridge,761,20.1199,0.0422,0.0458,-5.7840,-0.1043
7,ridge_no_system,761,18.3742,0.0386,0.0429,-3.6268,-0.1820
8,ridge_region,761,19.1582,0.0402,0.0441,-4.3428,-0.1471


In [10]:
print(f"Curso objetivo >= {RECENT_TARGET_FROM}")
display(recent_metrics(forecasts)[["model", "n_pairs", "wape", "wape_vs_naive"]].round(4))

Curso objetivo >= 2021


,model,n_pairs,wape,wape_vs_naive
0,drift,547,0.0441,-0.1016
1,gbm,547,0.0407,-0.1715
2,gbm_no_system,547,0.0390,-0.2059
3,gbm_region,547,0.0406,-0.1737
4,moving_average,547,0.0813,0.6562
5,naive,547,0.0491,0.0000
6,ridge,547,0.0380,-0.2269
7,ridge_no_system,547,0.0379,-0.2289
8,ridge_region,547,0.0383,-0.2198


## 7. Error por comunidad autónoma

Solo descriptivo: con 7 a 119 pares por comunidad no hay muestra para sacar conclusiones por región. La región
es la CCAA que trae el Excel del SIIU.

In [11]:
by_region = compute_metrics(add_region(main_forecasts, series), by=["region"])
shown = by_region[by_region["model"].isin(["naive", "ridge", "gbm"])]
display(shown[["region", "model", "n_pairs", "wape", "wape_vs_naive"]].round(3))

,region,model,n_pairs,wape,wape_vs_naive
1,Andalucía,gbm,119,0.034,0.135
2,Andalucía,naive,119,0.030,0.000
3,Andalucía,ridge,119,0.035,0.167
5,Aragón,gbm,28,0.035,-0.070
6,Aragón,naive,28,0.037,0.000
7,Aragón,ridge,28,0.039,0.036
9,Asturias (Principado de),gbm,14,0.022,-0.023
10,Asturias (Principado de),naive,14,0.023,0.000
11,Asturias (Principado de),ridge,14,0.035,0.545
13,Balears (Illes),gbm,7,0.026,-0.108


## 8. Comprobación contra el pipeline

El pipeline (`evaluate_models`) escribe `model_forecasts.csv` y `model_metrics.csv`. Se recalculan aquí con las
funciones del paquete y deben ser idénticos a los ficheros y a lo calculado arriba.

In [12]:
eligible = series[series["series_id"].isin(quality.loc[quality["eligible"], "series_id"])]
global_again = rolling_origin_global(eligible)
forecasts_again = pd.concat([rolling_origin_forecasts(eligible), global_again], ignore_index=True)
metrics_again = metrics_report(forecasts_again)

forecasts_disk = pd.read_csv(out_dir / "model_forecasts.csv")
metrics_disk = pd.read_csv(out_dir / "model_metrics.csv")
pd.testing.assert_frame_equal(global_again, forecasts_disk, check_dtype=False)
pd.testing.assert_frame_equal(metrics_again, metrics_disk, check_dtype=False)
pd.testing.assert_frame_equal(forecasts, forecasts_again)
pd.testing.assert_frame_equal(metrics, metrics_again)
print("El notebook y el pipeline producen exactamente lo mismo.")

El notebook y el pipeline producen exactamente lo mismo.
